In [41]:
import pandas as pd
import numpy as np
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import cross_val_score, StratifiedKFold,KFold,cross_validate
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
data_path = "..\dataset\insurance.csv"
df = pd.read_csv(data_path)
print("Dataset loaded successfully!")

Dataset loaded successfully!


In [3]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
class AdvancedFeatureEngineer:
    """
    Comprehensive feature engineering pipeline with domain knowledge
    """
    
    def __init__(self, verbose=True):
        self.verbose = verbose
        self.feature_names = []
        self.original_features = []
        
    def fit_transform(self, df):
        """
        Create all engineered features
        """
        if self.verbose:
            print("🔧 Starting Feature Engineering Pipeline...")
        
        df_feat = df.copy()
        self.original_features = df.columns.tolist()
        
        # Remove duplicate row if exists
        df_feat = df_feat.drop_duplicates()
        if self.verbose:
            print(f"✅ Removed {len(df) - len(df_feat)} duplicate rows")
        
        # ====================================================================
        # 1. DOMAIN-SPECIFIC FEATURES
        # ====================================================================
        if self.verbose:
            print("\n📊 Creating Domain-Specific Features...")
        
        # BMI Categories (WHO Classification)
        df_feat['bmi_category'] = pd.cut(
            df_feat['bmi'],
            bins=[0, 18.5, 25, 30, 35, 40, 100],
            labels=['Underweight', 'Normal', 'Overweight', 
                   'Obese_I', 'Obese_II', 'Obese_III']
        )
        
        # Simplified BMI risk
        df_feat['bmi_risk'] = df_feat['bmi_category'].map({
            'Underweight': 1,
            'Normal': 0,
            'Overweight': 1,
            'Obese_I': 2,
            'Obese_II': 3,
            'Obese_III': 4
        })
        
        # Age Groups (Insurance Industry Standard)
        df_feat['age_group'] = pd.cut(
            df_feat['age'],
            bins=[17, 25, 35, 45, 55, 65],
            labels=['18-25', '26-35', '36-45', '46-55', '56-64']
        )
        
        # Age risk score
        df_feat['age_risk'] = pd.cut(
            df_feat['age'],
            bins=[17, 30, 40, 50, 60, 65],
            labels=[1, 2, 3, 4, 5]
        ).astype(int)
        
        # Family size categories
        df_feat['family_size'] = df_feat['children'].map({
            0: 'single',
            1: 'small_family',
            2: 'small_family',
            3: 'large_family',
            4: 'large_family',
            5: 'large_family'
        })
        
        # # ====================================================================
        # # 2. INTERACTION FEATURES
        # # ====================================================================
        # if self.verbose:
        #     print("🔄 Creating Interaction Features...")
        
        # # Critical interaction: Smoker × BMI
        # df_feat['smoker_bmi'] = (df_feat['smoker'] == 'yes').astype(int) * df_feat['bmi']
        
        # # Smoker × Age
        # df_feat['smoker_age'] = (df_feat['smoker'] == 'yes').astype(int) * df_feat['age']
        
        # # Age × BMI
        # df_feat['age_bmi'] = df_feat['age'] * df_feat['bmi']
        
        # # Age × Children
        # df_feat['age_children'] = df_feat['age'] * df_feat['children']
        
        # # BMI × Children
        # df_feat['bmi_children'] = df_feat['bmi'] * df_feat['children']
        
        # # High-risk indicator (smoker with high BMI)
        # df_feat['high_risk'] = ((df_feat['smoker'] == 'yes') & 
        #                         (df_feat['bmi'] > 30)).astype(int)
        
        # # ====================================================================
        # # 3. POLYNOMIAL FEATURES
        # # ====================================================================
        # if self.verbose:
        #     print("📐 Creating Polynomial Features...")
        
        # # Age polynomials (non-linear relationship observed)
        # df_feat['age_squared'] = df_feat['age'] ** 2
        # df_feat['age_cubed'] = df_feat['age'] ** 3
        # df_feat['age_sqrt'] = np.sqrt(df_feat['age'])
        
        # # BMI polynomials
        # df_feat['bmi_squared'] = df_feat['bmi'] ** 2
        # df_feat['bmi_cubed'] = df_feat['bmi'] ** 3
        # df_feat['bmi_sqrt'] = np.sqrt(df_feat['bmi'])
        
        # ====================================================================
        # 4. LOGARITHMIC TRANSFORMATIONS
        # ====================================================================
        if self.verbose:
            print("📈 Creating Logarithmic Features...")
        
        df_feat['log_age'] = np.log1p(df_feat['age'])
        df_feat['log_bmi'] = np.log1p(df_feat['bmi'])
        df_feat['log_charges'] = np.log1p(df_feat['charges'])
        # df_feat['log_age_bmi'] = np.log1p(df_feat['age'] * df_feat['bmi'])
        
        # # ====================================================================
        # # 5. STATISTICAL FEATURES
        # # ====================================================================
        # if self.verbose:
        #     print("📊 Creating Statistical Features...")
        
        # # Z-scores (standardized features)
        # df_feat['age_zscore'] = (df_feat['age'] - df_feat['age'].mean()) / df_feat['age'].std()
        # df_feat['bmi_zscore'] = (df_feat['bmi'] - df_feat['bmi'].mean()) / df_feat['bmi'].std()
        
        # # Percentile ranks
        # df_feat['age_percentile'] = df_feat['age'].rank(pct=True)
        # df_feat['bmi_percentile'] = df_feat['bmi'].rank(pct=True)
        
        # # ====================================================================
        # # 6. COMPOSITE RISK SCORES
        # # ====================================================================
        # if self.verbose:
        #     print("🎯 Creating Composite Risk Scores...")
        
        # # Overall risk score (weighted combination)
        # df_feat['risk_score'] = (
        #     df_feat['age_risk'] * 0.25 +
        #     df_feat['bmi_risk'] * 0.20 +
        #     (df_feat['smoker'] == 'yes').astype(int) * 5 +
        #     df_feat['children'] * 0.1
        # )
        
        # # Health score (inverse of risk factors)
        # df_feat['health_score'] = (
        #     (100 - df_feat['age']) * 0.3 +
        #     (50 - abs(df_feat['bmi'] - 22)) * 0.3 +  # 22 is optimal BMI
        #     (df_feat['smoker'] == 'no').astype(int) * 40
        # )
        
        # # ====================================================================
        # # 7. REGIONAL ENCODING
        # # ====================================================================
        # if self.verbose:
        #     print("🗺️ Creating Regional Features...")
        
        # # Regional risk factors (based on healthcare costs by region)
        # region_risk_map = {
        #     'southeast': 1.2,
        #     'southwest': 1.0,
        #     'northeast': 1.1,
        #     'northwest': 0.9
        # }
        # df_feat['region_risk_factor'] = df_feat['region'].map(region_risk_map)
        
        # # ====================================================================
        # # 8. BINNING CONTINUOUS FEATURES
        # # ====================================================================
        # if self.verbose:
        #     print("📦 Creating Binned Features...")
        
        # # BMI bins
        # df_feat['bmi_bin_5'] = pd.cut(df_feat['bmi'], bins=5, labels=False)
        # df_feat['bmi_bin_10'] = pd.cut(df_feat['bmi'], bins=10, labels=False)
        
        # # Age bins
        # df_feat['age_bin_5'] = pd.cut(df_feat['age'], bins=5, labels=False)
        # df_feat['age_bin_10'] = pd.cut(df_feat['age'], bins=10, labels=False)
        
        # ====================================================================
        # FEATURE SUMMARY
        # ====================================================================
        new_features = [col for col in df_feat.columns if col not in self.original_features]
        self.feature_names = new_features
        
        if self.verbose:
            print("\n" + "="*80)
            print(f"✅ FEATURE ENGINEERING COMPLETE!")
            print(f"📊 Original features: {len(self.original_features)}")
            print(f"📊 New features created: {len(new_features)}")
            print(f"📊 Total features: {len(df_feat.columns)}")
            print("="*80)
        
        return df_feat
    
    def get_feature_names(self):
        """Return list of created feature names"""
        return self.feature_names

# Apply feature engineering
engineer = AdvancedFeatureEngineer(verbose=True)
df_engineered = engineer.fit_transform(df)

# Display sample of engineered features
print("\n📋 Sample of Engineered Dataset:")
display(df_engineered.head())

# Show new features created
print("\n📝 New Features Created:")
for i, feature in enumerate(engineer.get_feature_names(), 1):
    print(f"  {i:2d}. {feature}")

🔧 Starting Feature Engineering Pipeline...
✅ Removed 1 duplicate rows

📊 Creating Domain-Specific Features...
📈 Creating Logarithmic Features...

✅ FEATURE ENGINEERING COMPLETE!
📊 Original features: 7
📊 New features created: 8
📊 Total features: 15

📋 Sample of Engineered Dataset:


,age,sex,bmi,children,smoker,region,charges,bmi_category,bmi_risk,age_group,age_risk,family_size,log_age,log_bmi,log_charges
0,19,female,27.900,0,yes,southwest,16884.92400,Overweight,1,18-25,1,single,2.995732,3.363842,9.734236
1,18,male,33.770,1,no,southeast,1725.55230,Obese_I,2,18-25,1,small_family,2.944439,3.548755,7.453882
2,28,male,33.000,3,no,southeast,4449.46200,Obese_I,2,26-35,1,large_family,3.367296,3.526361,8.400763
3,33,male,22.705,0,no,northwest,21984.47061,Normal,0,26-35,2,single,3.526361,3.165686,9.998137
4,32,male,28.880,0,no,northwest,3866.85520,Overweight,1,26-35,2,single,3.496508,3.397189,8.260455



📝 New Features Created:
   1. bmi_category
   2. bmi_risk
   3. age_group
   4. age_risk
   5. family_size
   6. log_age
   7. log_bmi
   8. log_charges


In [5]:
df_engineered.head()

,age,sex,bmi,children,smoker,region,charges,bmi_category,bmi_risk,age_group,age_risk,family_size,log_age,log_bmi,log_charges
0,19,female,27.900,0,yes,southwest,16884.92400,Overweight,1,18-25,1,single,2.995732,3.363842,9.734236
1,18,male,33.770,1,no,southeast,1725.55230,Obese_I,2,18-25,1,small_family,2.944439,3.548755,7.453882
2,28,male,33.000,3,no,southeast,4449.46200,Obese_I,2,26-35,1,large_family,3.367296,3.526361,8.400763
3,33,male,22.705,0,no,northwest,21984.47061,Normal,0,26-35,2,single,3.526361,3.165686,9.998137
4,32,male,28.880,0,no,northwest,3866.85520,Overweight,1,26-35,2,single,3.496508,3.397189,8.260455


In [6]:
df_engineered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   age           1337 non-null   int64   
 1   sex           1337 non-null   object  
 2   bmi           1337 non-null   float64 
 3   children      1337 non-null   int64   
 4   smoker        1337 non-null   object  
 5   region        1337 non-null   object  
 6   charges       1337 non-null   float64 
 7   bmi_category  1337 non-null   category
 8   bmi_risk      1337 non-null   int64   
 9   age_group     1337 non-null   category
 10  age_risk      1337 non-null   int64   
 11  family_size   1337 non-null   object  
 12  log_age       1337 non-null   float64 
 13  log_bmi       1337 non-null   float64 
 14  log_charges   1337 non-null   float64 
dtypes: category(2), float64(5), int64(4), object(4)
memory usage: 149.3+ KB


In [43]:
X = df_engineered.drop(columns=['log_charges', 'age', 'bmi', 'charges'])
Y = df_engineered['log_charges']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stratify_cols = X['age_group'].astype(str) + '_' + \
                X['bmi_category'].astype(str) + '_' + \
                X['smoker'].astype(str)


X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=X['smoker']  # ← single argument!
)

category_selector = make_column_selector(pattern="cat")
object_selector = make_column_selector(dtype_include=object)
numerical_selector = ['children','bmi_risk','age_risk']

preprocessing_pipeline = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), category_selector),
    (OneHotEncoder(handle_unknown="ignore"), object_selector),
    (StandardScaler(), numerical_selector),
    remainder="drop")

model_pipeline = make_pipeline(
    preprocessing_pipeline,
    LinearRegression()
)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
results  = cross_validate(model_pipeline, X_train, y_train, cv=kfold,scoring='neg_mean_squared_error', return_estimator=True )
print(results.keys())
rmse_scores = np.sqrt(-results['test_score'])
print(f"RMSE scores: {rmse_scores}")
print(f"Mean RMSE: {rmse_scores.mean():.3f} (+/- {rmse_scores.std():.3f})")
# Access feature names from first fold's estimator
first_model = results['estimator'][0]
sed_model = results['estimator'][1]
feature_names = first_model.named_steps['columntransformer'].get_feature_names_out()
print(f"Features: {feature_names}")

# preprocessing_pipeline.get_feature_names_out()
# model_pipeline.get_feature_names_out()


dict_keys(['fit_time', 'score_time', 'estimator', 'test_score'])
RMSE scores: [0.470774   0.52646172 0.41661699 0.52483754 0.49144741]
Mean RMSE: 0.486 (+/- 0.041)
Features: ['onehotencoder-1__bmi_category_Normal'
 'onehotencoder-1__bmi_category_Obese_I'
 'onehotencoder-1__bmi_category_Obese_II'
 'onehotencoder-1__bmi_category_Obese_III'
 'onehotencoder-1__bmi_category_Overweight'
 'onehotencoder-1__bmi_category_Underweight' 'onehotencoder-2__sex_female'
 'onehotencoder-2__sex_male' 'onehotencoder-2__smoker_no'
 'onehotencoder-2__smoker_yes' 'onehotencoder-2__region_northeast'
 'onehotencoder-2__region_northwest' 'onehotencoder-2__region_southeast'
 'onehotencoder-2__region_southwest'
 'onehotencoder-2__family_size_large_family'
 'onehotencoder-2__family_size_single'
 'onehotencoder-2__family_size_small_family' 'standardscaler__children'
 'standardscaler__bmi_risk' 'standardscaler__age_risk']


In [44]:
model_pipeline.fit(X_train, y_train)

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder-1', ...), ('onehotencoder-2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [45]:
test_score = model_pipeline.score(X_test, y_test)
print(f"Test R² Score: {test_score:.3f}")

Test R² Score: 0.801


NotFittedError: This ColumnTransformer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

RMSE scores: [0.470774   0.52646172 0.41661699 0.52483754 0.49144741]
Mean RMSE: 0.486 (+/- 0.041)
